# 리포트 04 — 검출기: ECA · 거리도플러 · CFAR 교정

> ### ❓ 이 편이 답하는 질문
> **명목 Pfa 로 문턱을 세우면 실제 오경보율은 얼마가 되는가?**

### 결론
1. 이상적인 백색 잡음 맵에서는 명목 = 경험이다 — 배율 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ (셀 564,000,000 ⟨outputs/verify_cfar.json : white.48x24.rows[89].cells⟩개).
2. 실제 사슬(직접파 + ECA + Hann 창 + 정합필터)에서 명목 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 를 요구하면 WiFi 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 · LTE 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 · 5G 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 가 나온다.
3. 배율이 파형마다 다르므로, 교정하지 않은 3파형 비교는 **서로 다른 실제 오경보율에서** Pd 를 견주는 것이다. 즉 비교가 아니다.
4. 원인은 이웃 셀 상관이다 — Hann 창(도플러축)과 과표본(거리축). 둘 다 끄면 배율이 1.02 ⟨outputs/verify_cfar.json : control_whitened_mf_rect_NR100.op.rows[89].ratio⟩배로 돌아온다.
5. 교정표는 JSON 에 있고 검출 파이프라인이 그것을 직접 읽는다 (`src/passive_process.py:283`).

### ✅ 주장하는 것 / ❌ 주장하지 않는 것

| ✅ 이 편이 주장하는 것 | ❌ 이 편이 주장하지 않는 것 |
|---|---|
| **실제 사슬의 경험적 Pfa** — 파형 · 명목값마다 측정했다(파형 · 모드당 맵 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩개) | **절대 Pd 수치** — 표적 σ 에 걸려 있고, σ 는 리포트 02 의 측정 앵커에서 온다 |
| **CFAR 문턱 상수 자체는 이론과 일치한다** — 상대오차 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩ | **실외 클러터 환경의 Pfa** — 여기 배경은 잡음 + 광선추적 다중경로 모델이다 |
| **배율의 원인** — 대조군으로 확정했다(Hann 제거 · 백색화 정합필터) | **명목 1e-06 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[4].pfa_target_emp⟩ 교정** — 측정 구간 밖이라 외삽하지 않고 버렸다 |
| **교정표** — 목표 경험 Pfa → 줘야 할 명목 Pfa. 검출 코드가 이 JSON 을 소비한다 | **표적 위치 추정 성능** — 송수신 한 쌍은 관측가능하지 않다(§4) |
| **ECA 소거 깊이의 한계는 환경이지 알고리즘이 아니다** — §2 의 표 | **마이크로도플러** — future work. 이 편의 검출기는 표적을 점으로 본다 |

### 필요한 사전지식

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 리포트 03 | 세 조명원(WiFi · LTE · 5G NR)의 대역폭 · 기준신호 · 점유 모드 |
| 리포트 01 | 선행연구 census — 실외 실측 논문은 Pfa 를 통제할 수 없다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_observability.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/make_report04_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json`, `outputs/verify_eca.json`, `outputs/verify_observability.json` |
| 소요 | CFAR 측정이 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ (GPU 1장). ECA · 관측가능성 · 그림은 각각 수 분 이내. |
| 비고 | 맵 수는 `--maps` / `--white` 로 줄일 수 있다. 줄이면 신뢰구간이 넓어진다. |

---

## §1. 사슬 — 수신 신호가 판정이 되기까지

패시브 검출은 네 단계다. 각 단계는 앞 단계의 잔류물을 물려받는다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 수신 | 서베일런스 + 레퍼런스 2채널 | `src/passive_process.py:42` |
| 2. ECA | 직접파를 서베일런스에서 투영 제거 | `src/passive_process.py:93,124` |
| 3. 거리-도플러(CAF) | 레퍼런스와 지연 · 도플러 상관 | `src/passive_process.py:133` |
| 4. CA-CFAR | 이웃 셀로 문턱을 세우고 판정 | `src/passive_process.py:153` |

직접파는 방 안에서 가장 큰 신호다. 그 크기가 DNR 이고, 2단계가 지울 대상이다.

![passive detection chain](outputs/figures/report04_f1_chain.png)

**그림 1.** 수신 신호는 어떤 단계를 거쳐 검출 판정이 되는가?

### 사슬의 형상 — 파형이 정하는 것

거리 빈 수와 ECA 탭 수는 파형이 정한다. 도플러 빈은 세 파형 모두 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩개다(CPI 당 프레임 수).

| 파형 | DNR | ECA 탭 | 거리 빈 | PRF | Δf_d |
|---|---|---|---|---|---|
| WiFi 80MHz | 43.0 dB | 24 | 16 | 1000 Hz | 20.83 Hz |
| LTE 20MHz | 60.0 dB | 14 | 6 | 1000 Hz | 20.83 Hz |
| 5G NR 100MHz | 48.9 dB | 32 | 24 | 2000 Hz | 41.67 Hz |

출처 ⟨outputs/verify_eca.json : meta.setups⟩

## §2. ECA — 직접파를 얼마나 지우고, 무엇을 대가로 내는가

탭을 늘리면 소거가 깊어지다가 멈춘다. 멈추는 지점은 알고리즘이 아니라 환경이 정한다.

직접파만 있는 신호에 같은 소거기를 걸면 float64 한계까지 내려간다. 측정된 다중경로를 넣으면 아래 오른쪽 값에서 포화한다.

| 파형 | 직접파만 | 직접파 + 다중경로(포화) |
|---|---|---|
| WiFi | 202.7 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[1].rows[12].depth_dpi_db⟩ | 33.0 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[1].rows[12].depth_full_db⟩ |
| LTE | 219.9 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[2].rows[12].depth_dpi_db⟩ | 41.1 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[2].rows[12].depth_full_db⟩ |
| 5G | 232.3 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_dpi_db⟩ | 56.1 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_full_db⟩ |

![ECA cancellation depth vs taps](outputs/figures/report04_f2_eca_depth.png)

**그림 2.** ECA 소거 깊이의 한계를 정하는 것은 알고리즘인가 환경인가?

### ECA 의 대가 — 0-도플러 노치

ECA 는 지연만 다른 성분을 전부 지운다. 표적이 느리면 표적도 같이 지워진다.

노치 폭은 Δf_d 에 비례한다 — 3 dB 손실 지점은 f_d/Δf_d = 0.596 ⟨outputs/verify_eca.json : S4_target_loss[1].fd_3db_over_dfd⟩ 로 세 파형이 같다. λ 가 다르므로 속도 문턱은 파형마다 다르다 — CPI 프레임 48 ⟨outputs/verify_eca.json : S4_target_loss[4].M⟩개에서 WiFi 는 0.39 m/s ⟨outputs/verify_eca.json : S4_target_loss[4].v_3db_ms⟩ 아래를 못 본다(그림 3b).

정적 산란체는 ECA 뒤에서 죽은 파라미터다 — 클러터를 100 ⟨outputs/verify_eca.json : S5_clutter_dead.sweep[3].scale⟩배까지 키워도 SCR 변화폭은 3.5e-09 dB ⟨outputs/verify_eca.json : S5_clutter_dead.scr_span_db⟩ 다.

![zero-Doppler notch](outputs/figures/report04_f3_eca_notch.png)

**그림 3.** ECA 가 클러터와 함께 지우는 표적은 얼마나 느린 표적인가?

## §3. CFAR 교정 — 명목 Pfa 는 경험 Pfa 가 아니다

⭐ **이 절이 이 프로젝트에서 가장 방어하기 쉬운 결과다.** 실외 실측 논문은 Pfa 를 통제할 수 없고, OpenISAC 논문에는 CFAR · 오경보 · 검출확률이 등장하지 않는다(리포트 01 census). Pfa 교정은 통제된 시뮬레이션만 할 수 있는 일이다.

검출기 자체는 문제가 없다. 문턱 상수는 이론값과 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩ 안에서 같다. 평탄한 맵에서 오검출: 아니오 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.any_det_on_flat_map⟩.

이상적 백색 맵 500,000 ⟨outputs/verify_cfar.json : meta.n_maps_white⟩장에서 경험/명목은 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ 다. 실제 사슬에 걸면 그 눈금이 어긋난다. 이 절의 숫자는 전부 GPU 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ 측정이다.

![nominal vs empirical Pfa](outputs/figures/report04_f4_pfa.png)

**그림 4.** 명목 Pfa 를 요구하면 실제로는 몇 배의 오경보가 나오는가?

### 교정표 — 목표 경험 Pfa 를 얻으려면 명목값을 얼마로 줘야 하나

운용 명목값은 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 다. 아래 두 열은 그 값에서의 측정 결과다.

| 파형 | 경험/명목 배율 | 경험 1e-4 를 얻으려면 줄 명목 Pfa |
|---|---|---|
| WiFi | 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 | 6.270e-05 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ |
| LTE | 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 | 2.905e-05 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ |
| 5G | 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 | 6.460e-05 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ |

표의 두 열이 파형마다 다르다는 것이 §3 의 전부다. 같은 명목값을 줘도 LTE 는 5G 보다 훨씬 많이 울린다.

`src/passive_process.py:283` 이 이 JSON 을 직접 읽고, `pfa_nominal_for()`(`src/passive_process.py:338`)가 파형별 명목값을 돌려준다. 외삽 구간은 버린다.

In [ ]:
# 교정표를 실제로 소비하는 지점 — src/passive_process.py:283,338
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from passive_process import pfa_nominal_for

for std in ('wifi', 'lte', 'nr'):
    print(f'{std:4s}  경험 1e-4 목표 → 명목 {pfa_nominal_for(std, 1e-4):.3e}')

### 원인 — 검출기 결함이 아니라 셀 상관

CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 그 가정을 두 곳에서 깬다 — slow-time Hann 창이 도플러축을, 과표본이 거리축을 묶는다.

대조군이 원인을 확정한다(5G NR, 잡음 맵). 둘 다 끄면 눈금이 돌아온다.

| 대조군 | 경험/명목 |
|---|---|
| 기준 (Hann + 정합필터) | 1.25 ⟨outputs/verify_cfar.json : chain.NR100.noise.op.rows[89].ratio⟩ |
| Hann 제거 (rect 창) | 0.96 ⟨outputs/verify_cfar.json : control_rect_window_NR100.op.rows[89].ratio⟩ |
| 백색화 정합필터 (거리축 평탄) | 1.25 ⟨outputs/verify_cfar.json : control_whitened_mf_NR100.op.rows[89].ratio⟩ |
| 둘 다 제거 | 1.02 ⟨outputs/verify_cfar.json : control_whitened_mf_rect_NR100.op.rows[89].ratio⟩ |

![cell correlation controls](outputs/figures/report04_f5_cause.png)

**그림 5.** 명목과 경험 사이의 배율을 만드는 것은 무엇인가?

### 형상 규약 — 어기면 교정표가 무의미해진다

거리창은 ECA 탭 안에 있어야 하고, 0-도플러 행은 마스킹해야 한다(운용 폭 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩).

창을 256 ⟨outputs/verify_cfar.json : meta.n_range_wide⟩ 빈으로 넓히면 같은 명목 Pfa 에서 배율이 두 자릿수로 커진다. `check_detector_config()`(`src/passive_process.py:383`)가 이 조건을 검사한다.

| 파형 | 운용 창 | 넓은 창 |
|---|---|---|
| WiFi | 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 | 41.1 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.wide.rows[89].ratio⟩배 |
| LTE | 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 | 58.8 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.wide.rows[89].ratio⟩배 |
| 5G | 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 | 47.7 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.wide.rows[89].ratio⟩배 |

## §4. 분해능 · 정확도 · 관측가능성

분해능은 두 표적을 가르는 능력이고, 정확도는 한 표적을 얼마나 정밀히 찍는가다.

분해능은 대역폭이 정하며 SNR 로 좋아지지 않는다. 정확도는 SNR 이 정한다. 바이스태틱 규약은 ΔR_b = c/B 다(R_b = c·τ, 계수 2 없음).

![resolution vs accuracy](outputs/figures/report04_f6_resolution.png)

**그림 6.** 분해능과 정확도 중 대역폭이 정하는 것은 어느 쪽인가?

### 조명원별 셀 크기와 CRLB

5G 의 상시 기준신호(SSB)는 대역폭이 좁아 셀이 크다. 같은 반송파라도 기준신호를 PRS 로 바꾸면 셀이 작아진다.

도플러 분해능은 Δf_d = 1/T_CPI 다 — 아래 표의 형상은 T_CPI = 0.032 s ⟨outputs/verify_observability.json : cells[0].t_cpi⟩, Δf_d = 31.25 Hz ⟨outputs/verify_observability.json : cells[0].dfd_hz⟩. 그 아래 속도는 §2 의 노치가 먼저 지운다.

| 조명원 / 기준신호 | 기준 대역폭 | ΔR_b (분해능) | σ_Rb (정확도) | σ_fd |
|---|---|---|---|---|
| WiFi80 G1 (VHT-LTF) | 76.56 MHz | 3.75 m | 0.0162 m | 0.128 Hz |
| LTE20 G1 (CRS) | 17.98 MHz | 12.86 m | 0.0124 m | 0.022 Hz |
| 5G100 G1 (SSB) | 7.20 MHz | 39.20 m | 1.4901 m | 1.077 Hz |
| 5G100 G3 (PRS) | 98.28 MHz | 2.44 m | 0.0073 m | 0.073 Hz |

출처 ⟨outputs/verify_observability.json : cells⟩

### 관측가능성 — 송수신 한 쌍으로는 위치가 안 풀린다

판정: **NOT OBSERVABLE with one TX-RX pair ⟨outputs/verify_observability.json : summary.verdict⟩**. TX–RX 기저선 15.07 m ⟨outputs/verify_observability.json : meta.L_m⟩ 형상에서, 한 순간의 (R_b, f_d) 는 3차원 위치에 대해 랭크 2 ⟨outputs/verify_observability.json : summary.snapshot_fim_rank⟩ 다.

TX–RX 기저선을 축으로 표적을 돌려도 R_b 와 f_d 가 바뀌지 않는다 — 최대 변화 1.4e-14 m ⟨outputs/verify_observability.json : summary.exact_rotation_max_dRb_m⟩. 그 방향은 어떤 SNR 에서도, 어떤 관측시간에도 정보를 담지 않는다.

| 형상 | 유효 랭크 (/6) | 위치 RMS 오차 |
|---|---|---|
| 1RX (baseline) | 3 | 57.75 m |
| 2RX | 6 | 0.19 m |
| 1RX + AoA(1deg) | 6 | 0.12 m |
| 1RX + AoA(5deg) | 6 | 0.60 m |

출처 ⟨outputs/verify_observability.json : fixes⟩

![observability of one TX-RX pair](outputs/figures/report04_f7_observability.png)

**그림 7.** 송수신 한 쌍으로 표적 위치를 풀 수 있는가?

## §5. 이 편의 한계

| 아직 안 되어 있는 것 | 다음 사람이 이어받을 지점 |
|---|---|
| 명목 1e-06 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[4].pfa_target_emp⟩ 교정점이 없다 — 측정 구간 밖이라 버렸다 | `benchmark/verify_cfar.py --maps` 를 한 자릿수 올려 재측정. `calib_op_mask1.points` 의 `extrapolated` 가 False 가 되면 쓸 수 있다 |
| 교정이 잡음 + 광선추적 다중경로 위에서만 측정됐다 | 실외 클러터를 넣은 맵으로 같은 스윕을 돌려 배율이 유지되는지 확인 — 리포트 06 의 측정 계획이 그 조건을 정한다 |
| Pfa 는 교정됐지만 Pd 절대값은 표적 σ 에 걸려 있다 | 리포트 05 의 검출 결과는 리포트 02 의 측정 앵커 σ 위에서만 읽을 것 |
| 송수신 한 쌍 형상은 관측가능하지 않다 | 수신기 2대면 랭크 6 ⟨outputs/verify_observability.json : summary.fix_2rx_rank⟩ 로 복구되고 위치 RMS 가 0.19 m ⟨outputs/verify_observability.json : summary.fix_2rx_pos_rms_m⟩ 가 된다 — §4 표의 2RX 행 형상으로 검출 실험을 재설계 |
| 0-도플러 마스크 폭이 운용값 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩ 로 고정돼 있다 | 넓은 창에서 폭 3 은 과보정한다(배율 0.65 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.wide.rows[90].ratio⟩). 훈련셀에서도 그 행을 빼는 CFAR 변형을 만들고 재측정 — `src/passive_process.py:352` |
| 표적을 점 하나로 본다 — 마이크로도플러 성분이 없다 | 회전 블레이드는 별건의 검증이 필요하다. future work |